# Mimi-VC: Multi-Speaker Pre-trained Voice Conversion (V3f)

## What went wrong with V3e?
V3e added diversity loss (batch-entropy regularization) to fight mode collapse.
- With `.detach()` bug: diversity loss had zero gradient = pure cross-entropy = V3c
- After fix: diversity loss **made SECS worse** (0.823 → 0.725) without improving token diversity
- Still 37-41 unique tokens (vs 160 ground truth)

**Why batch-entropy diversity loss is counterproductive:**
- Different time steps already predict different tokens (~40 unique)
- Batch-average entropy (~4.9) already looks "moderately diverse"
- Diversity gradient spreads probability **within each position** (less confident)
- Result: lower accuracy, worse SECS, but NO new tokens added

## V3f fix: Focal Loss + Class-Frequency Weighting + Nucleus Sampling

| Component | V3e | V3f (this version) |
|-----------|-----|-----|
| Loss | Cross-entropy + batch-entropy diversity | **Focal loss** (gamma=2.0) |
| Class weighting | None | **Inverse sqrt frequency** (up-weight rare tokens) |
| Inference | Temperature + top-k sampling | **Nucleus (top-p) sampling** |
| Cross-speaker training | 50% FAISS pseudo-labels | **50% FAISS pseudo-labels** (kept) |
| CB0 augmentation | 30% span masking | **30% span masking** (kept) |

**Why focal loss works:**
- Standard CE: common tokens (p_t≈0.9) and rare tokens (p_t≈0.01) get similar gradient
- Focal loss: weight = (1-p_t)^gamma. Common tokens get weight 0.01, rare tokens get weight 0.98
- Model is **forced to learn the long tail** of the token distribution
- Combined with inverse-frequency class weights: rare tokens get even more gradient signal

**Pipeline:** Download LibriSpeech → Encode speakers + build FAISS indices + compute token frequencies
→ Pre-train V3f with focal loss → Fine-tune on target → Convert and compare

In [ ]:
# ==============================================================
# CELL 1: Install & Setup + Mount Google Drive
# ==============================================================
!pip install -q transformers torch torchaudio faiss-cpu librosa soundfile matplotlib resemblyzer datasets

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import numpy as np
import faiss
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import os, time, math, random
from collections import defaultdict, Counter
from transformers import MimiModel, AutoFeatureExtractor
from IPython.display import Audio as PlayAudio, display
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/mimi_vc_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {CKPT_DIR}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print('Loading Mimi...')
mimi = MimiModel.from_pretrained('kyutai/mimi').to(device).eval()
feat_ext = AutoFeatureExtractor.from_pretrained('kyutai/mimi')

SR = feat_ext.sampling_rate
N_CB = mimi.config.num_quantizers
CB_SIZE = mimi.config.codebook_size
CB_DIM = mimi.config.codebook_dim
FRAME_RATE = mimi.config.frame_rate
N_ACOU = N_CB - 1

print(f'SR={SR}, {N_CB} codebooks x {CB_SIZE}, dim={CB_DIM}, {FRAME_RATE}Hz')
print(f'Acoustic codebooks: {N_ACOU} (CB1..CB{N_CB-1})')

In [ ]:
# ==============================================================
# CELL 2: Core functions + per-speaker FAISS indices
# ==============================================================

def load_audio(path, target_sr=SR):
    wav, orig_sr = librosa.load(path, sr=None, mono=True)
    if orig_sr != target_sr:
        wav = librosa.resample(wav, orig_sr=orig_sr, target_sr=target_sr)
    return wav.astype(np.float32)

def encode(wav_np):
    inputs = feat_ext(raw_audio=wav_np, sampling_rate=SR, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = mimi.encode(inputs['input_values'], inputs.get('padding_mask'))
    return out.audio_codes.cpu()

def decode(codes):
    with torch.no_grad():
        out = mimi.decode(codes.to(device))
    return out.audio_values.cpu().squeeze().numpy()

def get_codebook_embeddings(cb_idx=0):
    q = mimi.quantizer
    if hasattr(q, 'semantic_residual_vector_quantizer'):
        n_sem = q.num_semantic_quantizers
        if cb_idx < n_sem:
            layer = q.semantic_residual_vector_quantizer.layers[cb_idx]
        else:
            layer = q.acoustic_residual_vector_quantizer.layers[cb_idx - n_sem]
    elif hasattr(q, 'layers'):
        layer = q.layers[cb_idx]
    else:
        raise RuntimeError(f'Unknown quantizer: {type(q).__name__}')
    for attr in ['codebook.embed', 'codebook.embed_sum', 'codebook.embedding.weight']:
        obj = layer
        try:
            for part in attr.split('.'):
                obj = getattr(obj, part)
            w = obj.detach().cpu().float()
            if w.dim() == 3: w = w.squeeze(0)
            if w.shape[0] >= CB_SIZE: return w
        except AttributeError:
            continue
    raise RuntimeError('Cannot find codebook embeddings')

# CB0 embeddings for FAISS retrieval
cb0_emb = get_codebook_embeddings(0).numpy()
print(f'CB0 embeddings: {cb0_emb.shape}')

N_LEARN = 6
print('Core functions ready.')

In [ ]:
# ==============================================================
# CELL 3: Download & encode LibriSpeech multi-speaker data
# ==============================================================
# Uses HuggingFace datasets for easy download
# train-clean-100: ~100 hours, 251 speakers

from datasets import load_dataset

print('Downloading LibriSpeech train-clean-100...')
print('(This may take a few minutes on first run)')
ds = load_dataset('librispeech_asr', 'clean', split='train.100', trust_remote_code=True)
print(f'Loaded {len(ds)} utterances')

# Group by speaker
speaker_utts = defaultdict(list)
for i, row in enumerate(ds):
    speaker_utts[row['speaker_id']].append(i)

speakers = sorted(speaker_utts.keys())
print(f'Speakers: {len(speakers)}')
print(f'Utterances per speaker: min={min(len(v) for v in speaker_utts.values())}, '
      f'max={max(len(v) for v in speaker_utts.values())}, '
      f'mean={np.mean([len(v) for v in speaker_utts.values()]):.0f}')

In [ ]:
# ==============================================================
# CELL 4: Encode speakers + FAISS indices + token frequencies
# ==============================================================

N_SPEAKERS = 40        # number of speakers to use
UTTS_PER_SPEAKER = 20  # utterances per speaker
MAX_AUDIO_SEC = 15     # max seconds per utterance

# Pick speakers with most utterances
speakers_sorted = sorted(speakers, key=lambda s: len(speaker_utts[s]), reverse=True)
selected_speakers = speakers_sorted[:N_SPEAKERS]

print(f'Encoding {N_SPEAKERS} speakers x {UTTS_PER_SPEAKER} utterances...')
print(f'Max {MAX_AUDIO_SEC}s per utterance')

speaker_data = {}
speaker_raw_audio = {}  # store raw audio for resemblyzer embeddings
t0 = time.time()

for si, spk_id in enumerate(selected_speakers):
    utt_indices = speaker_utts[spk_id][:UTTS_PER_SPEAKER]
    all_codes = []
    boundaries = []
    total_frames = 0
    spk_wavs = []

    for idx in utt_indices:
        row = ds[idx]
        wav = np.array(row['audio']['array'], dtype=np.float32)
        orig_sr = row['audio']['sampling_rate']
        if orig_sr != SR:
            wav = librosa.resample(wav, orig_sr=orig_sr, target_sr=SR)
        wav = wav[:int(MAX_AUDIO_SEC * SR)]
        if len(wav) < SR:
            continue
        spk_wavs.append(wav)
        codes = encode(wav)  # [1, N_CB, T]
        T = codes.shape[2]
        boundaries.append((total_frames, total_frames + T))
        all_codes.append(codes[0])  # [N_CB, T]
        total_frames += T

    if all_codes:
        speaker_data[spk_id] = {
            'codes': torch.cat(all_codes, dim=1),  # [N_CB, total_T]
            'boundaries': boundaries,
        }
        speaker_raw_audio[spk_id] = np.concatenate(spk_wavs)

    if (si + 1) % 10 == 0:
        print(f'  {si+1}/{N_SPEAKERS} speakers encoded | {time.time()-t0:.0f}s')

encode_time = time.time() - t0
total_frames_all = sum(v['codes'].shape[1] for v in speaker_data.values())
print(f'\nDone: {len(speaker_data)} speakers, {total_frames_all} total frames '
      f'({total_frames_all/FRAME_RATE:.0f}s) in {encode_time:.0f}s')

# === Build per-speaker FAISS indices for cross-speaker training ===
print('\nBuilding per-speaker FAISS indices...')
K_FAISS = 8
ALPHA_FAISS = 0.5
speaker_faiss = {}  # spk_id -> {'index': faiss.Index, 'full_codes': np.array [T, N_CB]}

for spk_id, sdata in speaker_data.items():
    codes = sdata['codes']  # [N_CB, T]
    sem_tokens = codes[0].numpy()  # [T]
    full_codes = codes.numpy().T   # [T, N_CB]

    # Build FAISS index on CB0 embeddings
    emb = cb0_emb[sem_tokens].astype(np.float32).copy()
    faiss.normalize_L2(emb)
    idx = faiss.IndexFlatIP(cb0_emb.shape[1])
    idx.add(emb)

    speaker_faiss[spk_id] = {
        'index': idx,
        'full_codes': full_codes,
    }

print(f'Built FAISS indices for {len(speaker_faiss)} speakers')
print(f'Average index size: {np.mean([sf["index"].ntotal for sf in speaker_faiss.values()]):.0f} vectors')

# === Compute per-codebook token frequencies for focal loss class weights ===
print('\nComputing token frequencies for class-balanced focal loss...')
cb_token_counts = torch.zeros(N_LEARN, CB_SIZE, dtype=torch.long)
for spk_id, sdata in speaker_data.items():
    codes = sdata['codes']
    for cb_i in range(N_LEARN):
        tokens = codes[1 + cb_i]
        cb_token_counts[cb_i] += torch.bincount(tokens, minlength=CB_SIZE)

cb_class_weights = []
for cb_i in range(N_LEARN):
    freq = cb_token_counts[cb_i].float()
    freq = freq / freq.sum()
    w = 1.0 / (freq.sqrt() + 1e-6)
    w = w / w.mean()
    w = w.clamp(max=20.0)
    cb_class_weights.append(w)
    used = (cb_token_counts[cb_i] > 0).sum().item()
    print(f'  CB{cb_i+1}: {used} tokens appear in training data')

cb_class_weights = torch.stack(cb_class_weights)  # [N_LEARN, CB_SIZE]
print(f'Class weights shape: {cb_class_weights.shape}')

In [ ]:
# ==============================================================
# CELL 5: Compute speaker embeddings via Resemblyzer (d-vectors)
# ==============================================================
# Resemblyzer produces 256-dim speaker embeddings trained on speaker verification
# Much stronger speaker identity signal than averaging CB0 embeddings

from resemblyzer import VoiceEncoder, preprocess_wav

print('Loading Resemblyzer speaker encoder...')
spk_enc = VoiceEncoder(device=device)

SPK_EMB_DIM = 256  # resemblyzer output dim

speaker_embeddings = {}
speaker_ids = sorted(speaker_data.keys())
spk_to_idx = {s: i for i, s in enumerate(speaker_ids)}

print(f'Computing d-vector embeddings for {len(speaker_ids)} speakers...')
for spk_id in speaker_ids:
    wav16k = librosa.resample(speaker_raw_audio[spk_id], orig_sr=SR, target_sr=16000)
    speaker_embeddings[spk_id] = spk_enc.embed_utterance(wav16k)

# Build matrix for quick lookup
spk_emb_matrix = np.stack([speaker_embeddings[s] for s in speaker_ids])
spk_emb_tensor = torch.tensor(spk_emb_matrix, dtype=torch.float32).to(device)

print(f'Speaker embeddings: {spk_emb_matrix.shape}')
print(f'Inter-speaker cosine similarity:')
norms = spk_emb_matrix / (np.linalg.norm(spk_emb_matrix, axis=1, keepdims=True) + 1e-8)
sim_matrix = norms @ norms.T
np.fill_diagonal(sim_matrix, 0)
print(f'  Mean: {sim_matrix[sim_matrix > 0].mean():.4f}')
print(f'  Min:  {sim_matrix[sim_matrix > 0].min():.4f}')
print(f'  Max:  {sim_matrix.max():.4f}')

# Free raw audio to save memory
del speaker_raw_audio
import gc; gc.collect()
print('Speaker embeddings ready (raw audio freed).')

In [ ]:
# ==============================================================
# CELL 6: AcousticTranslatorV3f — Focal Loss + Nucleus Sampling
# ==============================================================
#
# V3e failed: batch-entropy diversity loss was COUNTERPRODUCTIVE.
# - Made SECS worse (0.823 -> 0.725) without improving token diversity
# - Still 37-41 unique tokens (batch avg entropy looked "diverse" but
#   each position was still peaked on the same small set of tokens)
#
# V3f fix:
#   1. FOCAL LOSS: (1-p_t)^gamma down-weights easy/common tokens
#      Forces model to learn the long tail of the token distribution
#   2. CLASS FREQUENCY WEIGHTS: inverse sqrt frequency balancing
#      Rare tokens get more gradient signal
#   3. NUCLEUS SAMPLING at inference: top-p instead of top-k
#   4. Cross-speaker training with FAISS pseudo-labels (kept from V3e)
#   5. CB0 span masking 30% + speaker noise std=0.3 (kept)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_len=8192):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        t = torch.arange(max_len, dtype=inv_freq.dtype)
        freqs = torch.outer(t, inv_freq)
        self.register_buffer('cos_cached', freqs.cos()[None, None, :, :], persistent=False)
        self.register_buffer('sin_cached', freqs.sin()[None, None, :, :], persistent=False)

    def forward(self, x):
        T = x.shape[2]
        return self.cos_cached[:, :, :T, :], self.sin_cached[:, :, :T, :]

def apply_rope(q, k, cos, sin):
    def rotate(x, c, s):
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        return torch.cat([x1*c - x2*s, x2*c + x1*s], dim=-1)
    return rotate(q, cos, sin), rotate(k, cos, sin)

class RoPETransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout=0.15):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.norm1 = nn.LayerNorm(d_model)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.o_proj = nn.Linear(d_model, d_model)
        self.rope = RotaryEmbedding(self.head_dim)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, ff_dim), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(ff_dim, d_model), nn.Dropout(dropout))
        self.dropout = nn.Dropout(dropout)

    def _reshape(self, x, B, T):
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x):
        """Bidirectional attention — NO causal mask."""
        B, T, _ = x.shape
        h = self.norm1(x)
        q = self._reshape(self.q_proj(h), B, T)
        k = self._reshape(self.k_proj(h), B, T)
        v = self._reshape(self.v_proj(h), B, T)
        cos, sin = self.rope(q)
        q, k = apply_rope(q, k, cos, sin)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, self.d_model)
        x = x + self.dropout(self.o_proj(out))
        x = x + self.ffn(self.norm2(x))
        return x


class FiLMLayer(nn.Module):
    def __init__(self, d_model, cond_dim):
        super().__init__()
        self.proj = nn.Linear(cond_dim, d_model * 2)

    def forward(self, x, cond):
        gamma_beta = self.proj(cond).unsqueeze(1)
        gamma, beta = gamma_beta.chunk(2, dim=-1)
        return x * (1 + gamma) + beta


class AcousticTranslatorV3f(nn.Module):
    """
    V3f: Focal loss + class-frequency weighting + nucleus sampling.

    Batch-entropy diversity loss is counterproductive — it makes predictions
    less confident (hurting SECS) without increasing the set of tokens used.
    Focal loss directly addresses mode collapse by forcing the model to learn
    rare/hard tokens in the long tail of the token distribution.
    """
    def __init__(self, n_vocab=2048, n_cb=6, d_model=512, n_heads=8,
                 n_enc_layers=6, n_dec_layers=6, spk_emb_dim=256,
                 dropout=0.15, cb0_noise_rate=0.30, spk_noise_std=0.3):
        super().__init__()
        self.n_cb = n_cb
        self.n_vocab = n_vocab
        self.d_model = d_model
        self.cb0_noise_rate = cb0_noise_rate
        self.spk_noise_std = spk_noise_std
        ff_dim = d_model * 4

        self.sem_embed = nn.Embedding(n_vocab, d_model)
        self.cb_embed = nn.Embedding(n_cb, d_model)

        self.spk_proj = nn.Sequential(
            nn.Linear(spk_emb_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.film_layers = nn.ModuleList([
            FiLMLayer(d_model, d_model) for _ in range(n_dec_layers)
        ])

        self.encoder = nn.ModuleList([
            RoPETransformerLayer(d_model, n_heads, ff_dim, dropout)
            for _ in range(n_enc_layers)
        ])
        self.enc_norm = nn.LayerNorm(d_model)

        self.decoder = nn.ModuleList([
            RoPETransformerLayer(d_model, n_heads, ff_dim, dropout)
            for _ in range(n_dec_layers)
        ])
        self.dec_norm = nn.LayerNorm(d_model)

        # Output heads: n_vocab logits
        self.output_heads = nn.ModuleList([
            nn.Linear(d_model, n_vocab) for _ in range(n_cb)
        ])

    def _augment_cb0(self, sem_tokens):
        """Span masking: corrupt ~30% of CB0 tokens in contiguous chunks."""
        if not self.training or self.cb0_noise_rate <= 0:
            return sem_tokens
        B, T = sem_tokens.shape
        result = sem_tokens.clone()
        target_corrupt = int(T * self.cb0_noise_rate)
        for b in range(B):
            corrupted = 0
            while corrupted < target_corrupt:
                span_len = random.randint(3, min(10, target_corrupt - corrupted + 3))
                span_len = min(span_len, T)
                start = random.randint(0, T - span_len)
                result[b, start:start+span_len] = torch.randint(
                    0, self.n_vocab, (span_len,), device=sem_tokens.device)
                corrupted += span_len
        return result

    def _augment_spk(self, spk_emb):
        """Add Gaussian noise to speaker embedding during training."""
        if not self.training or self.spk_noise_std <= 0:
            return spk_emb
        return spk_emb + torch.randn_like(spk_emb) * self.spk_noise_std

    def encode_semantic(self, sem_tokens):
        sem_tokens = self._augment_cb0(sem_tokens)
        x = self.sem_embed(sem_tokens)
        for layer in self.encoder:
            x = layer(x)
        return self.enc_norm(x)

    def forward(self, sem_tokens, tgt_spk_emb):
        """Returns logits [B, n_cb, T, n_vocab]."""
        B, T = sem_tokens.shape
        sem_enc = self.encode_semantic(sem_tokens)
        spk_cond = self.spk_proj(self._augment_spk(tgt_spk_emb))

        # Parallel CB: batch all codebooks together
        sem_exp = sem_enc.unsqueeze(1).expand(B, self.n_cb, T, self.d_model)
        sem_exp = sem_exp.reshape(B * self.n_cb, T, self.d_model)

        cb_ids = torch.arange(self.n_cb, device=sem_tokens.device)
        cb_ids = cb_ids.unsqueeze(0).expand(B, self.n_cb).reshape(B * self.n_cb)
        cb_emb = self.cb_embed(cb_ids).unsqueeze(1)
        h = sem_exp + cb_emb

        spk_exp = spk_cond.unsqueeze(1).expand(B, self.n_cb, self.d_model)
        spk_exp = spk_exp.reshape(B * self.n_cb, self.d_model)

        for layer, film in zip(self.decoder, self.film_layers):
            h = layer(h)
            h = film(h, spk_exp)
        h = self.dec_norm(h)

        h = h.reshape(B, self.n_cb, T, self.d_model)

        all_logits = []
        for cb_i in range(self.n_cb):
            all_logits.append(self.output_heads[cb_i](h[:, cb_i]))  # [B, T, n_vocab]
        return torch.stack(all_logits, dim=1)  # [B, n_cb, T, n_vocab]

    def compute_loss(self, logits, target_tokens, cb_weights, class_weights=None, focal_gamma=2.0):
        """Focal loss with class-frequency weighting.

        Focal: -alpha_t * (1 - p_t)^gamma * log(p_t)
        - Common tokens (p_t~0.9): weight ~ 0.01 -> tiny gradient
        - Rare tokens (p_t~0.01): weight ~ 0.98 -> large gradient
        - Forces model to learn the long tail of the token distribution
        """
        B, n_cb, T, V = logits.shape
        total_loss = 0

        for cb_i in range(n_cb):
            cb_logits = logits[:, cb_i].reshape(-1, V)  # [B*T, V]
            cb_targets = target_tokens[:, cb_i].reshape(-1)  # [B*T]

            # Log-probabilities
            log_probs = F.log_softmax(cb_logits.float(), dim=-1)  # [B*T, V]

            # Probability of correct class
            target_log_probs = log_probs.gather(1, cb_targets.unsqueeze(1)).squeeze(1)
            target_probs = target_log_probs.exp()  # p_t

            # Focal weight: (1 - p_t)^gamma — detached (standard practice)
            focal_weight = (1.0 - target_probs.detach()) ** focal_gamma

            # Class frequency weight (if provided)
            if class_weights is not None:
                alpha_t = class_weights[cb_i].to(cb_logits.device)
                alpha_t = alpha_t.gather(0, cb_targets)
                focal_weight = focal_weight * alpha_t

            # Focal loss
            loss = -(focal_weight * target_log_probs).mean()

            total_loss += cb_weights[cb_i] * loss

        return total_loss / n_cb

    @torch.no_grad()
    def generate(self, sem_tokens, tgt_spk_emb, temperature=1.0, top_p=0.92):
        """Generate tokens with nucleus (top-p) sampling.

        Nucleus sampling keeps the smallest set of tokens whose cumulative
        probability exceeds top_p. Preserves diversity in the tail.
        """
        self.eval()
        logits = self.forward(sem_tokens, tgt_spk_emb)
        logits = logits / temperature

        B, n_cb, T, V = logits.shape
        tokens = torch.zeros(B, n_cb, T, dtype=torch.long, device=logits.device)

        for cb_i in range(self.n_cb):
            cb_logits = logits[0, cb_i]  # [T, V]
            probs = F.softmax(cb_logits.float(), dim=-1)  # [T, V]

            # Sort descending
            sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)

            # Cumulative probability
            cum_probs = torch.cumsum(sorted_probs, dim=-1)

            # Mask tokens where cumulative prob (excl. current) exceeds top_p
            mask = (cum_probs - sorted_probs) >= top_p
            sorted_probs[mask] = 0.0

            # Renormalize
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)

            # Sample one token per position
            sampled = torch.multinomial(sorted_probs, 1).squeeze(-1)  # [T]
            tokens[0, cb_i] = sorted_indices.gather(1, sampled.unsqueeze(1)).squeeze(1)

        return tokens


print(f'AcousticTranslatorV3f: CB0 + speaker_emb -> CB1-{N_LEARN}')
print(f'Output: {CB_SIZE}-class logits')
print(f'Loss: focal loss (gamma=2.0) + class-frequency weighting')
print(f'Inference: nucleus sampling (top-p)')
print(f'Training: 50% self-reconstruction + 50% cross-speaker (FAISS pseudo-labels)')
print(f'CB0 noise: 30% span masking, speaker noise: std=0.3')

In [ ]:
# ==============================================================
# CELL 7: Mixed pre-training (V3f — self-recon + cross-speaker)
# ==============================================================
# KEY CHANGE: 50% self-reconstruction + 50% cross-speaker training.
# Cross-speaker targets come from FAISS retrieval into another speaker's
# codebook space. This teaches the model what cross-speaker conversion
# actually looks like — something V3a-V3d never learned.
#
# V3f: Replace diversity loss with focal loss + class-frequency weighting.
# Diversity loss was counterproductive (worse SECS, same token count).

D_MODEL = 512
N_HEADS = 8
SEQ_LEN = 128
BATCH_SIZE = 48         # A100: 48-64 (was 24 on T4)
LR = 2e-4
WARMUP_STEPS = 1000
N_PRETRAIN_STEPS = 30000
SAVE_EVERY = 5000
FOCAL_GAMMA = 2.0  # focusing parameter: 0 = standard CE, 2 = strong focus on rare tokens
CROSS_SPEAKER_RATIO = 0.5  # fraction of batch that is cross-speaker

cb_weights = torch.tensor([2.0**(-(i)/2) for i in range(N_LEARN)], device=device)
cb_weights = cb_weights / cb_weights.sum() * N_LEARN

print(f'=== MULTI-SPEAKER PRE-TRAINING (V3f — CROSS-SPEAKER) ===')
print(f'Self-reconstruction + cross-speaker FAISS distillation')
print(f'Steps: {N_PRETRAIN_STEPS}, Batch: {BATCH_SIZE}, Seq: {SEQ_LEN}')
print(f'Cross-speaker ratio: {CROSS_SPEAKER_RATIO*100:.0f}%')
print(f'LOSS: focal loss (gamma={FOCAL_GAMMA}) + class-frequency weighting')
print(f'AUGMENTATION: CB0 span masking 30%, speaker noise std=0.3')
print(f'Checkpoints: {CKPT_DIR}')

model = AcousticTranslatorV3f(
    n_vocab=CB_SIZE, n_cb=N_LEARN, d_model=D_MODEL, n_heads=N_HEADS,
    n_enc_layers=6, n_dec_layers=6, spk_emb_dim=SPK_EMB_DIM,
    dropout=0.15, cb0_noise_rate=0.30, spk_noise_std=0.3
).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params: {n_params/1e6:.1f}M (trainable)')

use_compile = hasattr(torch, 'compile') and device == 'cuda'
if use_compile:
    try:
        model = torch.compile(model)
        print('torch.compile: ENABLED')
    except Exception:
        use_compile = False
        print('torch.compile: disabled')

use_amp = device == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
amp_dtype = torch.float16
print(f'AMP: {"ENABLED" if use_amp else "disabled"}')

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / (N_PRETRAIN_STEPS - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

# Resume from Drive
start_step = 0
resume_path = os.path.join(CKPT_DIR, 'v3f_pretrain_latest.pt')
if os.path.exists(resume_path):
    ckpt = torch.load(resume_path, weights_only=False, map_location=device)
    state = ckpt['model_state']
    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    clean = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
    raw_model.load_state_dict(clean)
    opt.load_state_dict(ckpt['opt_state'])
    start_step = ckpt['step']
    for _ in range(start_step):
        sched.step()
    print(f'\n>> RESUMED from step {start_step}')
    if start_step >= N_PRETRAIN_STEPS:
        print(f'   Pre-training already complete. Loading best and skipping.')
else:
    print()

spk_list = list(speaker_data.keys())


def faiss_retrieve_tokens(src_sem_tokens, tgt_spk_id):
    """Use FAISS to map source CB0 tokens into target speaker's codebook space.
    Returns acoustic tokens [N_LEARN, T] as pseudo ground truth."""
    T = len(src_sem_tokens)
    sf = speaker_faiss[tgt_spk_id]

    # Look up CB0 embeddings for source tokens
    src_emb = cb0_emb[src_sem_tokens].astype(np.float32).copy()
    faiss.normalize_L2(src_emb)

    # FAISS search with temporal smoothing
    dists_k, indices_k = sf['index'].search(src_emb, K_FAISS)

    selected = np.zeros(T, dtype=np.int64)
    selected[0] = indices_k[0, 0]
    for t in range(1, T):
        cands = indices_k[t]
        combined = ALPHA_FAISS * dists_k[t] + (1 - ALPHA_FAISS) / (
            1 + np.abs(cands - selected[t-1]).astype(np.float32))
        selected[t] = cands[np.argmax(combined)]

    # Get the full codebook vectors at retrieved positions
    full_codes = sf['full_codes']  # [T_tgt, N_CB]
    retrieved = full_codes[selected]  # [T, N_CB]

    # Return CB1 through CB(N_LEARN) as target
    return torch.tensor(retrieved[:, 1:1+N_LEARN].T, dtype=torch.long)  # [N_LEARN, T]


def sample_mixed_batch(batch_size, seq_len, cross_speaker_ratio):
    """Sample a batch with mix of self-reconstruction and cross-speaker examples.

    Self-reconstruction: speaker A's CB0 + speaker A's embedding → speaker A's CB1-6
    Cross-speaker: speaker A's CB0 + speaker B's embedding → FAISS(A→B) CB1-6
    """
    n_cross = int(batch_size * cross_speaker_ratio)
    n_self = batch_size - n_cross

    sem_batch, tgt_acou_batch, spk_batch = [], [], []

    # Self-reconstruction examples
    for _ in range(n_self):
        spk = random.choice(spk_list)
        codes = speaker_data[spk]['codes']
        T = codes.shape[1]
        start = random.randint(0, max(0, T - seq_len))
        crop = codes[:, start:start+seq_len]
        if crop.shape[1] < seq_len:
            crop = F.pad(crop, (0, seq_len - crop.shape[1]))
        sem_batch.append(crop[0])
        tgt_acou_batch.append(crop[1:1+N_LEARN])
        spk_batch.append(torch.tensor(speaker_embeddings[spk], dtype=torch.float32))

    # Cross-speaker examples (FAISS pseudo-labels)
    for _ in range(n_cross):
        src_spk = random.choice(spk_list)
        tgt_spk = random.choice(spk_list)
        while tgt_spk == src_spk:
            tgt_spk = random.choice(spk_list)

        codes = speaker_data[src_spk]['codes']
        T = codes.shape[1]
        start = random.randint(0, max(0, T - seq_len))
        crop = codes[:, start:start+seq_len]
        if crop.shape[1] < seq_len:
            crop = F.pad(crop, (0, seq_len - crop.shape[1]))

        sem_tokens = crop[0]  # source CB0
        # FAISS retrieval: map source CB0 into target speaker's space
        pseudo_acou = faiss_retrieve_tokens(sem_tokens.numpy(), tgt_spk)  # [N_LEARN, T]

        sem_batch.append(sem_tokens)
        tgt_acou_batch.append(pseudo_acou)
        spk_batch.append(torch.tensor(speaker_embeddings[tgt_spk], dtype=torch.float32))

    return (
        torch.stack(sem_batch).to(device),
        torch.stack(tgt_acou_batch).to(device),
        torch.stack(spk_batch).to(device),
    )


all_losses = []
per_cb_accs = {i: [] for i in range(N_LEARN)}
model.train()
t0 = time.time()
log_interval = max(1, N_PRETRAIN_STEPS // 30)
best_loss = float('inf')

best_path = os.path.join(CKPT_DIR, 'v3f_pretrain_best.pt')
if os.path.exists(best_path):
    best_ckpt = torch.load(best_path, weights_only=False, map_location='cpu')
    best_loss = best_ckpt.get('loss', float('inf'))
    del best_ckpt
    print(f'Existing best loss on Drive: {best_loss:.4f}')

for step in range(start_step, N_PRETRAIN_STEPS):
    sem_b, tgt_acou_b, spk_b = sample_mixed_batch(BATCH_SIZE, SEQ_LEN, CROSS_SPEAKER_RATIO)

    with torch.amp.autocast('cuda', enabled=use_amp, dtype=amp_dtype):
        logits = model(sem_b, spk_b)  # [B, N_LEARN, T, CB_SIZE]
        loss = model.compute_loss(logits, tgt_acou_b, cb_weights, cb_class_weights.to(device), FOCAL_GAMMA)

    # Compute accuracy for logging
    if (step + 1) % log_interval == 0:
        with torch.no_grad():
            for cb_i in range(N_LEARN):
                pred_tokens = logits[:, cb_i].float().argmax(-1).reshape(-1)
                gt_tokens = tgt_acou_b[:, cb_i].reshape(-1)
                acc = (pred_tokens == gt_tokens).float().mean().item()
                per_cb_accs[cb_i].append(acc * 100)

    opt.zero_grad()
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt)
    scaler.update()
    sched.step()
    all_losses.append(loss.item())

    if (step + 1) % log_interval == 0:
        cb_str = ' '.join([f'CB{i+1}:{per_cb_accs[i][-1]:.0f}%' for i in range(N_LEARN)])
        elapsed = time.time() - t0
        steps_done = step + 1 - start_step
        sps = steps_done / elapsed
        eta = (N_PRETRAIN_STEPS - step - 1) / sps
        cur_lr = sched.get_last_lr()[0]
        print(f'  Step {step+1:5d}/{N_PRETRAIN_STEPS} | Loss:{loss.item():.4f} | {cb_str} | LR:{cur_lr:.1e} | {sps:.1f} step/s | ETA {eta:.0f}s')

    if (step + 1) % SAVE_EVERY == 0:
        avg_loss = np.mean(all_losses[-SAVE_EVERY:])
        raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
        ckpt_data = {
            'step': step + 1,
            'model_state': raw_model.state_dict(),
            'opt_state': opt.state_dict(),
            'loss': avg_loss,
        }
        torch.save(ckpt_data, os.path.join(CKPT_DIR, f'v3f_pretrain_step{step+1}.pt'))
        torch.save(ckpt_data, os.path.join(CKPT_DIR, 'v3f_pretrain_latest.pt'))

        tag = ''
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(ckpt_data, best_path)
            tag = ' ** BEST'
        print(f'  >> Saved to Drive: step {step+1} (avg loss: {avg_loss:.4f}){tag}')

pretrain_time = time.time() - t0

if os.path.exists(best_path):
    best_ckpt = torch.load(best_path, weights_only=False, map_location=device)
    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    raw_model.load_state_dict(best_ckpt['model_state'])
    print(f'\nLoaded best checkpoint (loss: {best_loss:.4f})')
model.eval()

if all_losses:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
    axes[0].plot(all_losses, linewidth=0.5, color='#7030A0')
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
    axes[0].set_title(f'Pre-training ({N_PRETRAIN_STEPS} steps, {pretrain_time/60:.0f}min)')
    axes[0].grid(True, alpha=0.3)

    colors = plt.cm.tab10(np.linspace(0, 1, N_LEARN))
    for i in range(N_LEARN):
        axes[1].plot(per_cb_accs[i], label=f'CB{i+1}', color=colors[i])
    axes[1].set_xlabel('Log point'); axes[1].set_ylabel('Accuracy %')
    axes[1].set_title('Per-CB Accuracy (mixed self-recon + cross-speaker)')
    axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
    plt.savefig(os.path.join(CKPT_DIR, 'v3f_pretrain_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()

    total_steps = N_PRETRAIN_STEPS - start_step
    sps = total_steps / pretrain_time if pretrain_time > 0 else 0
    print(f'\nPre-training done: {pretrain_time:.0f}s ({pretrain_time/60:.1f}min)')
    print(f'Throughput: {sps:.1f} steps/s')
    for i in range(N_LEARN):
        print(f'  CB{i+1}: {per_cb_accs[i][-1]:.1f}%')
    print(f'\nNote: accuracy is lower on cross-speaker examples (FAISS targets are noisy).')
    print(f'This is expected. The key metric is inference-time token diversity + SECS.')
else:
    print(f'\nPre-training already complete (loaded from checkpoint).')

In [ ]:
# ==============================================================
# CELL 8: Upload TARGET speaker audio & encode
# ==============================================================

from google.colab import files
print('Upload TARGET speaker audio file(s):')
uploaded_target = files.upload()

CHUNK_SEC = 30
chunk_samples = int(CHUNK_SEC * SR)

target_wavs = []
for fname, data in uploaded_target.items():
    tmp = f'/tmp/target_{fname}'
    with open(tmp, 'wb') as f:
        f.write(data)
    wav = load_audio(tmp)
    print(f'  {fname}: {len(wav)/SR:.1f}s')
    target_wavs.append(wav)

target_wav_concat = np.concatenate(target_wavs)
total_target_dur = len(target_wav_concat) / SR
print(f'Total target: {total_target_dur:.1f}s')

# Encode target with Mimi
print('\nEncoding target speaker...')
tgt_chunks = []
for wav in target_wavs:
    for i in range(0, len(wav), chunk_samples):
        chunk = wav[i:i+chunk_samples]
        if len(chunk) > SR:
            tgt_chunks.append(chunk)

tgt_codes_list = [encode(c) for c in tgt_chunks]
all_tgt_codes = torch.cat(tgt_codes_list, dim=2)
T_tgt = all_tgt_codes.shape[2]
print(f'Target: {T_tgt} frames ({T_tgt/FRAME_RATE:.1f}s)')

# Compute target speaker embedding via Resemblyzer (same as training)
print('Computing target d-vector...')
tgt_wav_16k = librosa.resample(target_wav_concat, orig_sr=SR, target_sr=16000)
tgt_spk_emb = spk_enc.embed_utterance(tgt_wav_16k)  # [256]
tgt_spk_emb_t = torch.tensor(tgt_spk_emb, dtype=torch.float32).to(device)
print(f'Target speaker embedding: {tgt_spk_emb_t.shape}')

# Build FAISS index for retrieval (used for CB{N_LEARN+1}+)
print('\nBuilding FAISS index...')
all_sem = all_tgt_codes[0, 0].numpy()
all_full = all_tgt_codes[0].numpy().T  # [T_tgt, N_CB]
tgt_emb = cb0_emb[all_sem].astype(np.float32).copy()
faiss.normalize_L2(tgt_emb)
faiss_index = faiss.IndexFlatIP(cb0_emb.shape[1])
faiss_index.add(tgt_emb)
print(f'FAISS index: {faiss_index.ntotal} vectors')

display(PlayAudio(target_wav_concat[:int(SR*15)], rate=SR))

In [ ]:
# ==============================================================
# CELL 9: Fine-tune on target speaker (V3f with cross-entropy)
# ==============================================================
# Fine-tune with self-reconstruction on target speaker data.
# The model has already learned cross-speaker conversion in pre-training.
# Fine-tuning adapts to the specific target speaker's voice characteristics.
# Lower diversity weight than pre-training to allow specialization.

FT_STEPS = 3000
FT_LR = 5e-5
FT_BATCH = 32           # A100: 32-48 (was 16 on T4)
FT_FOCAL_GAMMA = 1.5   # lower than pre-training (2.0) — allow target specialization

print(f'=== FINE-TUNING ON TARGET SPEAKER (V3f) ===')
print(f'Target data: {T_tgt} frames ({T_tgt/FRAME_RATE:.1f}s)')
print(f'Focal gamma: {FT_FOCAL_GAMMA} (vs {FOCAL_GAMMA} in pre-training)')

# Check for existing fine-tuned checkpoint
ft_ckpt_path = os.path.join(CKPT_DIR, 'v3f_finetuned.pt')
skip_ft = False
if os.path.exists(ft_ckpt_path):
    ft_ckpt = torch.load(ft_ckpt_path, weights_only=False, map_location=device)
    saved_emb = ft_ckpt.get('tgt_spk_emb')
    if saved_emb is not None and np.allclose(saved_emb, tgt_spk_emb, atol=1e-5):
        raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
        clean = {k.replace('_orig_mod.', ''): v for k, v in ft_ckpt['model_state'].items()}
        raw_model.load_state_dict(clean)
        model.eval()
        skip_ft = True
        print(f'\nLoaded fine-tuned checkpoint (same target speaker). Skipping fine-tuning.')
        print(f'To retrain, delete: {ft_ckpt_path}')
    else:
        print(f'\nFine-tuned checkpoint exists but for different target speaker. Retraining...')
    del ft_ckpt

if not skip_ft:
    print(f'Steps: {FT_STEPS}, Batch: {FT_BATCH}, LR: {FT_LR}')
    print(f'Loss: focal loss (gamma={FT_FOCAL_GAMMA}) + class-frequency weighting')
    print(f'Augmentation ON: CB0 span masking + speaker noise')

    ft_sem = all_tgt_codes[0, 0]
    ft_acou = all_tgt_codes[0, 1:1+N_LEARN]

    ft_opt = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=0.01)
    ft_sched = torch.optim.lr_scheduler.CosineAnnealingLR(ft_opt, FT_STEPS)
    ft_scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    ft_losses = []
    ft_accs = {i: [] for i in range(N_LEARN)}
    model.train()
    t0 = time.time()
    ft_log = max(1, FT_STEPS // 20)

    for step in range(FT_STEPS):
        starts = torch.randint(0, max(1, T_tgt - SEQ_LEN), (FT_BATCH,))
        sem_b = torch.stack([ft_sem[s:s+SEQ_LEN] for s in starts]).to(device)
        acou_b = torch.stack([ft_acou[:, s:s+SEQ_LEN] for s in starts]).to(device)
        spk_b = tgt_spk_emb_t.unsqueeze(0).expand(FT_BATCH, -1)

        with torch.amp.autocast('cuda', enabled=use_amp, dtype=amp_dtype):
            logits = model(sem_b, spk_b)
            loss = model.compute_loss(logits, acou_b, cb_weights, cb_class_weights.to(device), FT_FOCAL_GAMMA)

        if (step + 1) % ft_log == 0:
            with torch.no_grad():
                for cb_i in range(N_LEARN):
                    pred_tokens = logits[:, cb_i].float().argmax(-1).reshape(-1)
                    gt_tokens = acou_b[:, cb_i].reshape(-1)
                    acc = (pred_tokens == gt_tokens).float().mean().item()
                    ft_accs[cb_i].append(acc * 100)

        ft_opt.zero_grad()
        ft_scaler.scale(loss).backward()
        ft_scaler.unscale_(ft_opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        ft_scaler.step(ft_opt)
        ft_scaler.update()
        ft_sched.step()
        ft_losses.append(loss.item())

        if (step + 1) % ft_log == 0:
            cb_str = ' '.join([f'CB{i+1}:{ft_accs[i][-1]:.0f}%' for i in range(N_LEARN)])
            print(f'  FT {step+1:4d}/{FT_STEPS} | Loss:{loss.item():.4f} | {cb_str} | {time.time()-t0:.0f}s')

    ft_time = time.time() - t0
    model.eval()

    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    torch.save({
        'model_state': raw_model.state_dict(),
        'config': {'n_vocab': CB_SIZE, 'n_cb': N_LEARN, 'd_model': D_MODEL,
                   'n_heads': N_HEADS, 'spk_emb_dim': SPK_EMB_DIM},
        'tgt_spk_emb': tgt_spk_emb,
    }, ft_ckpt_path)

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(ft_losses, linewidth=0.8, color='#C55A11')
    ax.set_xlabel('Step'); ax.set_ylabel('Loss')
    ax.set_title(f'Fine-tuning ({FT_STEPS} steps, {ft_time:.0f}s)')
    ax.grid(True, alpha=0.3)
    plt.show()

    print(f'\nFine-tuning done ({ft_time:.0f}s). Saved to Drive.')
    for i in range(N_LEARN):
        print(f'  CB{i+1}: {ft_accs[i][-1]:.1f}%')

In [ ]:
# ==============================================================
# CELL 10: Upload SOURCE audio
# ==============================================================

print('Upload SOURCE audio file(s):')
uploaded_source = files.upload()

source_wavs = []
for fname, data in uploaded_source.items():
    tmp = f'/tmp/source_{fname}'
    with open(tmp, 'wb') as f:
        f.write(data)
    wav = load_audio(tmp)
    print(f'  {fname}: {len(wav)/SR:.1f}s')
    source_wavs.append(wav)

source_wav = np.concatenate(source_wavs)
source_dur = len(source_wav) / SR
print(f'Total source: {source_dur:.1f}s')

src_codes = encode(source_wav)
T_src = src_codes.shape[2]
print(f'Source: {T_src} frames')

display(PlayAudio(source_wav, rate=SR))

In [ ]:
# ==============================================================
# CELL 11: Voice conversion — all methods
# ==============================================================

print('='*70)
print('VOICE CONVERSION — V3f vs RETRIEVAL')
print('='*70)

# --- 1. FAISS+Smooth retrieval baseline ---
print('\n1. FAISS+Smooth (retrieval baseline)...')
src_sem_np = src_codes[0, 0].numpy()
src_emb_q = cb0_emb[src_sem_np].astype(np.float32).copy()
faiss.normalize_L2(src_emb_q)
K = 8; ALPHA = 0.5
dists_k, indices_k = faiss_index.search(src_emb_q, K)

selected = np.zeros(T_src, dtype=np.int64)
selected[0] = indices_k[0, 0]
for t in range(1, T_src):
    cands = indices_k[t]
    combined = ALPHA * dists_k[t] + (1-ALPHA) / (1 + np.abs(cands - selected[t-1]).astype(np.float32))
    selected[t] = cands[np.argmax(combined)]

ret_codes = src_codes.clone()
for t in range(T_src):
    ret_codes[0, :, t] = torch.tensor(all_full[selected[t]])
ret_wav = decode(ret_codes)
print(f'  Done. Avg top-1 sim: {dists_k[:, 0].mean():.4f}')

# --- 2. V3f (nucleus sampling with different top_p) ---
print('\n2. V3f nucleus sampling...')
src_sem_d = src_codes[0, 0:1].to(device)  # [1, T]
spk_d = tgt_spk_emb_t.unsqueeze(0)         # [1, 256]

v3_results = {}
for top_p in [0.85, 0.92, 0.98]:
    torch.manual_seed(42)
    pred = model.generate(src_sem_d, spk_d, temperature=1.0, top_p=top_p)
    vc = src_codes.clone()
    vc[0, 1:1+N_LEARN] = pred[0].cpu()
    # Fill remaining CBs from retrieval
    for t in range(T_src):
        vc[0, 1+N_LEARN:, t] = torch.tensor(all_full[selected[t]][1+N_LEARN:])
    w = decode(vc)
    v3_results[top_p] = (vc, w)
    print(f'  V3f top_p={top_p}: done')

# --- 3. V3f + keep source CB0 ---
print('\n3. V3f (keep source CB0)...')
v3_keepcb0 = src_codes.clone()
v3_keepcb0[0, 1:1+N_LEARN] = v3_results[0.92][0][0, 1:1+N_LEARN]
for t in range(T_src):
    v3_keepcb0[0, 1+N_LEARN:, t] = torch.tensor(all_full[selected[t]][1+N_LEARN:])
v3_keepcb0_wav = decode(v3_keepcb0)

# --- Playback ---
print(f'\n{"="*70}')
print('PLAYBACK')
print(f'{"="*70}')

playback = [
    ('Source (original)', source_wav),
    ('FAISS+Smooth (retrieval)', ret_wav),
    ('V3f (p=0.85)', v3_results[0.85][1]),
    ('V3f (p=0.92)', v3_results[0.92][1]),
    ('V3f (p=0.98)', v3_results[0.98][1]),
    ('V3f keep CB0 (p=0.92)', v3_keepcb0_wav),
    ('Target reference', target_wav_concat[:int(SR*15)]),
]
for label, wav in playback:
    print(f'\n{label}:')
    display(PlayAudio(wav, rate=SR))


In [ ]:
# ==============================================================
# CELL 12: Diagnostics & metrics
# ==============================================================

print('='*70)
print('DIAGNOSTICS — V3f')
print('='*70)

# --- Token distribution ---
print('\n--- Token distribution: V3f on source audio ---')
v3_pred_codes = v3_results[0.92][0][0, 1:1+N_LEARN, :].numpy()
for cb_i in range(N_LEARN):
    pred_tokens = v3_pred_codes[cb_i]
    n_unique = len(set(pred_tokens.tolist()))
    c = Counter(pred_tokens.tolist())
    probs = np.array(list(c.values())) / len(pred_tokens)
    entropy = -np.sum(probs * np.log2(probs + 1e-10))
    print(f'  CB{cb_i+1}: {n_unique:4d} unique tokens, entropy={entropy:.2f}')

print(f'\n  V3e had: 37-41 unique tokens, entropy ~4.9 (diversity loss counterproductive)')
print(f'  V3c had: 40 unique tokens, entropy ~4.89 (self-recon mode collapse)')
print(f'  Ground truth: ~160 unique tokens, entropy ~7.1')
print(f'  V3f target: 80-160 unique tokens, entropy 6-7')

# --- Per-CB change rate ---
print(f'\n--- Per-codebook change rate (vs source) ---')
N_SHOW = min(10, N_CB)
src_t = src_codes[0, :, :].numpy()
print(f'{"Method":<25}', end='')
for cb in range(N_SHOW):
    print(f' CB{cb:>2}', end='')
print()
print('-' * (25 + N_SHOW * 5))

for name, codes in [
    ('FAISS+Smooth', ret_codes),
    ('V3f (p=0.92)', v3_results[0.92][0]),
    ('V3f keep CB0', v3_keepcb0),
]:
    c = codes[0, :, :].numpy()
    print(f'{name:<25}', end='')
    for cb in range(N_SHOW):
        rate = np.mean(c[cb] != src_t[cb]) * 100
        print(f' {rate:3.0f}%', end='')
    print()

# --- Objective metrics ---
print(f'\n--- Objective metrics ---')

def compute_secs(a, b):
    a16 = librosa.resample(a.astype(np.float32), orig_sr=SR, target_sr=16000)
    b16 = librosa.resample(b.astype(np.float32), orig_sr=SR, target_sr=16000)
    ea = spk_enc.embed_utterance(a16)
    eb = spk_enc.embed_utterance(b16)
    return float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb)))

def compute_mcd(ref, syn):
    r = librosa.feature.mfcc(y=ref.astype(np.float32), sr=SR, n_mfcc=13)
    s = librosa.feature.mfcc(y=syn.astype(np.float32), sr=SR, n_mfcc=13)
    L = min(r.shape[1], s.shape[1])
    if L == 0: return None
    diff = r[1:, :L] - s[1:, :L]
    return float(np.mean(np.sqrt(2.0 * np.sum(diff**2, axis=0))))

tgt_ref_wav = target_wav_concat[:int(SR*15)]
src_tgt_secs = compute_secs(source_wav, tgt_ref_wav)

print(f'  Source vs Target baseline: SECS={src_tgt_secs:.4f}')
print(f'\n{"Method":<30} {"SECS(->tgt)":>12} {"SECS(->src)":>12} {"MCD":>8}')
print('-'*65)

eval_methods = [
    ('FAISS+Smooth (retrieval)', ret_wav),
    ('V3f (p=0.85)', v3_results[0.85][1]),
    ('V3f (p=0.92)', v3_results[0.92][1]),
    ('V3f (p=0.98)', v3_results[0.98][1]),
    ('V3f keep CB0 (p=0.92)', v3_keepcb0_wav),
]

for name, wav in eval_methods:
    secs_tgt = compute_secs(wav, tgt_ref_wav)
    secs_src = compute_secs(wav, source_wav)
    mcd = compute_mcd(source_wav, wav)
    print(f'{name:<30} {secs_tgt:>12.4f} {secs_src:>12.4f} {mcd:>8.2f}')

print(f'\n  SECS(->tgt): higher = more like target voice')
print(f'  SECS(->src): higher = still sounds like source')
print(f'  MCD: lower = less distortion')


In [ ]:
# ==============================================================
# CELL 13: Visual comparison
# ==============================================================

L_v = min(T_src, 200)

# Token heatmaps
fig, axes = plt.subplots(5, 1, figsize=(16, 14), constrained_layout=True)
items = [
    ('Source', src_codes[0, :, :L_v]),
    ('FAISS+Smooth', ret_codes[0, :, :L_v]),
    ('V3f (p=0.92)', v3_results[0.92][0][0, :, :L_v]),
    ('V3f keep CB0', v3_keepcb0[0, :, :L_v]),
    ('Target ref', all_tgt_codes[0, :, :L_v]),
]
for ax, (title, codes) in zip(axes, items):
    im = ax.imshow(codes.numpy(), aspect='auto', cmap='viridis', interpolation='nearest')
    ax.set_ylabel('CB'); ax.set_title(title, fontsize=10)
    ax.set_yticks(range(0, N_CB, max(1, N_CB//8)))
    plt.colorbar(im, ax=ax, shrink=0.5)
axes[-1].set_xlabel('Frame')
plt.suptitle('Token Heatmaps — V3f vs Retrieval', fontsize=13)
plt.savefig('/content/heatmaps_v3f.png', dpi=150, bbox_inches='tight')
plt.show()

# Mel spectrograms
fig, axes = plt.subplots(5, 1, figsize=(14, 12), constrained_layout=True)
for ax, (lbl, w) in zip(axes, [
    ('Source', source_wav),
    ('FAISS+Smooth', ret_wav),
    ('V3f (p=0.92)', v3_results[0.92][1]),
    ('V3f keep CB0', v3_keepcb0_wav),
    ('Target ref', target_wav_concat[:int(SR*15)]),
]):
    mel = librosa.feature.melspectrogram(y=w, sr=SR, n_mels=80, fmax=8000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(mel_db, sr=SR, x_axis='time', y_axis='mel', ax=ax, fmax=8000)
    ax.set_title(lbl, fontsize=10)
axes[-1].set_xlabel('Time (s)')
plt.suptitle('Mel Spectrograms — V3f vs Retrieval', fontsize=13)
plt.savefig('/content/spectrograms_v3f.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ==============================================================
# CELL 14: Save & download (to Google Drive + local)
# ==============================================================

save_items = [
    ('faiss_smooth.wav', ret_wav),
    ('v3f_p085.wav', v3_results[0.85][1]),
    ('v3f_p092.wav', v3_results[0.92][1]),
    ('v3f_p098.wav', v3_results[0.98][1]),
    ('v3f_keepcb0.wav', v3_keepcb0_wav),
]

for fname, wav in save_items:
    sf.write(f'/content/{fname}', wav, SR)
    sf.write(os.path.join(CKPT_DIR, fname), wav, SR)
    print(f'Saved: {fname}')

# Save final model checkpoint to Drive
raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
torch.save({
    'model_state': raw_model.state_dict(),
    'config': {
        'n_vocab': CB_SIZE, 'n_cb': N_LEARN, 'd_model': D_MODEL,
        'n_heads': N_HEADS, 'spk_emb_dim': SPK_EMB_DIM,
    },
    'tgt_spk_emb': tgt_spk_emb,
}, os.path.join(CKPT_DIR, 'v3f_finetuned.pt'))
print(f'Saved: {CKPT_DIR}/v3f_finetuned.pt')

# Also zip and download
!cd /content && zip -q v3f_results.zip *.wav *.png
from google.colab import files
files.download('/content/v3f_results.zip')
print(f'\nAll results also saved to: {CKPT_DIR}')
